/content
Enter GitHub Token: ··········
Cloning into 'IF5153_NLP_Final_Projects'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 223 (delta 77), reused 183 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 7.10 MiB | 19.64 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/IF5153_NLP_Final_Projects


# ABSA Experiment
Experiment to make extract keywords from the task 1 and passing the keywords result to task 3

Author: Raihan Muhammad Daffa<br/>
NIM: 18223140

## Import Modules

In [2]:
import os
import sys
import random
import numpy as np
import re
import json
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, set_seed


SEED = 42

def reproducible(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reproducible(SEED)

## GPU Utilization Check

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: NVIDIA A100-SXM4-80GB


## Helper

In [4]:
def build_prompt(review_text):
    return(
        "### Instruction: Extract the key aspects and their sentiments from the following game review"
        "You MUST format the output strictly as Aspects: "
        "[category:aspect:sentiment, ...]\n\n"
        f"### Input: Review: \"{review_text}\"\n\n"
        "### Response:"
    )

def parse_aspect_tuples(text):
    clean_text = re.sub(r"Aspects\s*:\s*\[?", "", text, flags=re.IGNORECASE).replace("]", "")
    matches = re.findall(r"([A-Za-z0-9_/]+):([^:\],]+):([A-Za-z]+)", clean_text)
    return set((cat.strip().lower(), asp.strip().lower(), sent.strip().lower()) for cat, asp, sent in matches)

def evaluate_model(model, tokenizer, test_data, experiment_name, batch_size=16):
    model.eval()
    exact_matches = 0
    total_p, total_r, total_f1 = 0.0, 0.0, 0.0
    n = len(test_data)

    for i in range(0, n, batch_size):
        batch = test_data[i : i + batch_size]
        texts = [item["input_text"] for item in batch]
        golds = [item["target_text"] for item in batch]

        inputs = tokenizer(texts, return_tensors="pt", max_length=1024, padding=True, truncation=True).to(device)
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=4,
                repetition_penalty=1.0,
                length_penalty=1.5,
                early_stopping=True
            )

        preds = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        for pred_text, gold_text in zip(preds, golds):
            pred_triplets = parse_aspect_tuples(pred_text)
            gold_triplets = parse_aspect_tuples(gold_text)

            if pred_triplets == gold_triplets and len(gold_triplets) > 0:
                exact_matches += 1

            intersection = len(pred_triplets & gold_triplets)

            if pred_triplets:
                prec = intersection / len(pred_triplets)
            else:
                prec = 0.0

            if gold_triplets:
                rec = intersection / len(gold_triplets)
            else:
                rec = 0.0

            if prec + rec > 0:
                f1 = 2 * (prec * rec) / (prec + rec)
            else:
                f1 = 0.0

            total_p += prec
            total_r += rec
            total_f1 += f1

    results = {
        "Experiment": experiment_name,
        "Exact Match (%)": round((exact_matches / n) * 100, 2),
        "Precision (%)": round((total_p / n) * 100, 2),
        "Recall (%)": round((total_r / n) * 100, 2),
        "F1-Score (%)": round((total_f1 / n) * 100, 2)
    }
    return results

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, bart_tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, bart_tokenizer.pad_token_id)
    decoded_preds = bart_tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens=True)
    exact_matches = 0

    total_p, total_r, total_f1 = 0.0, 0.0, 0.0
    n = len(decoded_labels)

    for pred_text, gold_text in zip(decoded_preds, decoded_labels):
        pred_triplets = parse_aspect_tuples(pred_text)
        gold_triplets = parse_aspect_tuples(gold_text)
        if pred_triplets == gold_triplets and len(gold_triplets) > 0:
            exact_matches += 1
        intersection = len(pred_triplets & gold_triplets)
        prec = intersection / len(pred_triplets) if pred_triplets else 0.0
        rec = intersection / len(gold_triplets) if gold_triplets else 0.0
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        total_p += prec
        total_r += rec
        total_f1 += f1

    return {
        "em": round((exact_matches / n) * 100, 2),
        "precision": round((total_p / n) * 100, 2),
        "recall": round((total_r / n) * 100, 2),
        "f1": round((total_f1 / n) * 100, 2),
    }

all_expe_results = []

In [5]:
class ABSADataset(Dataset):
    def __init__(self, data_pairs, tokenizer, max_input_len=1024, max_target_len=128):
        self.data_pairs = data_pairs
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        item = self.data_pairs[idx]
        model_inputs = self.tokenizer(item["input_text"], max_length=self.max_input_len, truncation=True)
        labels = self.tokenizer(text_target=item["target_text"], max_length=self.max_target_len, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

## Load the dataset

In [6]:
with open(JSON_OUTPUT_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

data_pairs = []
for game_id, game_data in raw_data.items():
    for review_pair in game_data.get("review_pairs", []):
        if review_pair["target_label"] != "Aspects: []":
            data_pairs.append({
                "game": game_data.get("app_name", f"Game {game_id}"),
                "input_text": build_prompt(review_pair["review"]),
                "target_text": review_pair["target_label"]
            })

## Split the data

In [7]:
n_data = len(data_pairs)
n_train = int(0.70 * n_data)
n_val = int(0.85 * n_data)

train_data = data_pairs[:n_train]
val_data = data_pairs[n_train:n_val]
test_data = data_pairs[n_val:]

print(f"Total samples: {n_data} | Train: {len(train_data)} |  Validation: {len(val_data)} | Test: {len(test_data)}")

Total samples: 12081 | Train: 8456 |  Validation: 1812 | Test: 1813


## Experiment 1 (Zero-Base Shot)

In [8]:
base_model_name = "facebook/bart-base"
bart_tokenizer = BartTokenizer.from_pretrained(base_model_name)
bart_zero_shot_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [9]:
res_zero_shot = evaluate_model(bart_zero_shot_model, bart_tokenizer, test_data, "Zero-Shot (Raw BART)")
all_expe_results.append(res_zero_shot)
print(pd.DataFrame([res_zero_shot]))

             Experiment  Exact Match (%)  Precision (%)  Recall (%)  \
0  Zero-Shot (Raw BART)              0.0            0.0         0.0   

   F1-Score (%)  
0           0.0  


In [10]:
del bart_zero_shot_model
torch.cuda.empty_cache()

## Experiment 2 (Task-Adapted BART)

In [11]:
bart_task_adapted_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)
bart_train_dataset = ABSADataset(train_data, bart_tokenizer)
bart_val_dataset = ABSADataset(val_data, bart_tokenizer)
bart_data_collator = DataCollatorForSeq2Seq(tokenizer=bart_tokenizer,model=bart_task_adapted_model)

bart_task_adapted_training_arguments = Seq2SeqTrainingArguments(
    output_dir="models/bart_4epochs",
    num_train_epochs=4,
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=10,
    logging_strategy="epoch",
    eval_strategy="epoch",
    generation_num_beams=2,
    lr_scheduler_type="cosine",
    save_strategy="no",
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

bart_task_adapted_trainer = Seq2SeqTrainer(
    model=bart_task_adapted_model,
    args=bart_task_adapted_training_arguments,
    train_dataset=bart_train_dataset,
    eval_dataset=bart_val_dataset,
    data_collator=bart_data_collator,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [12]:
bart_task_adapted_trainer.train()

Epoch,Training Loss,Validation Loss,Em,Precision,Recall,F1
1,3.642925,0.137918,38.020000,58.640000,52.500000,53.890000
2,1.096938,0.109900,42.380000,62.690000,56.230000,57.850000
3,0.829883,0.097333,47.020000,68.540000,60.920000,63.000000
4,0.678161,0.093381,48.230000,70.390000,62.150000,64.510000


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


TrainOutput(global_step=1060, training_loss=1.5619769690171728, metrics={'train_runtime': 802.1904, 'train_samples_per_second': 42.165, 'train_steps_per_second': 1.321, 'total_flos': 8698355024486400.0, 'train_loss': 1.5619769690171728, 'epoch': 4.0})

In [13]:
bart_task_adapted_evaluation = evaluate_model(bart_task_adapted_model, bart_tokenizer, test_data, "Task-Adapted (BART - 4 Epochs)")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [14]:
all_expe_results.append(bart_task_adapted_evaluation)
print(pd.DataFrame([bart_task_adapted_evaluation]))

                       Experiment  Exact Match (%)  Precision (%)  Recall (%)  \
0  Task-Adapted (BART - 4 Epochs)             57.8          70.33       71.55   

   F1-Score (%)  
0          70.2  


## Experiments 3

In [15]:
bart_hyperparametered_task_adapted_model = BartForConditionalGeneration.from_pretrained(base_model_name).to(device)

bart_hyperparametered_task_adapted_training_arguments = Seq2SeqTrainingArguments(
    output_dir="models/bart_6epochs",
    num_train_epochs=6,
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    lr_scheduler_type="cosine",
    save_strategy="no",
    logging_strategy="epoch",
    eval_strategy="epoch",
    generation_num_beams=2,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

bart_hyperparametered_task_adapted_trainer = Seq2SeqTrainer(
    model=bart_hyperparametered_task_adapted_model,
    args=bart_hyperparametered_task_adapted_training_arguments,
    train_dataset=bart_train_dataset,
    eval_dataset=bart_val_dataset,
    data_collator=bart_data_collator,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [16]:
bart_hyperparametered_task_adapted_trainer.train()

Epoch,Training Loss,Validation Loss,Em,Precision,Recall,F1
1,4.868444,0.144439,37.470000,58.220000,51.560000,53.210000
2,1.165522,0.120419,40.620000,58.660000,52.720000,54.270000
3,0.871732,0.103601,46.410000,67.270000,60.030000,62.020000
4,0.660760,0.088765,50.880000,73.430000,65.040000,67.410000
5,0.524931,0.086264,52.370000,75.520000,66.450000,69.100000
6,0.449453,0.086658,52.040000,74.920000,65.930000,68.550000


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


TrainOutput(global_step=1590, training_loss=1.4234736220641706, metrics={'train_runtime': 1199.6765, 'train_samples_per_second': 42.291, 'train_steps_per_second': 1.325, 'total_flos': 1.305851136565248e+16, 'train_loss': 1.4234736220641706, 'epoch': 6.0})

In [17]:
bart_hyperparametered_task_adapted_evaluation = evaluate_model(bart_hyperparametered_task_adapted_model, bart_tokenizer, test_data, "Hyperparametered Task-Adapted (BART - 6 Epochs)")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [18]:
all_expe_results.append(bart_hyperparametered_task_adapted_evaluation)
print(pd.DataFrame([bart_hyperparametered_task_adapted_evaluation]))

                                        Experiment  Exact Match (%)  \
0  Hyperparametered Task-Adapted (BART - 6 Epochs)            63.71   

   Precision (%)  Recall (%)  F1-Score (%)  
0          74.99       76.14         74.85  


## Evaluation

In [19]:
summary_df = pd.DataFrame(all_expe_results)
display(summary_df)

,Experiment,Exact Match (%),Precision (%),Recall (%),F1-Score (%)
0,Zero-Shot (Raw BART),0.00,0.00,0.00,0.00
1,Task-Adapted (BART - 4 Epochs),57.80,70.33,71.55,70.20
2,Hyperparametered Task-Adapted (BART - 6 Epochs),63.71,74.99,76.14,74.85


## Error Analysis

In [20]:
best_model = bart_hyperparametered_task_adapted_model
best_model.eval()

for sample in test_data[:8]:
    inputs = bart_tokenizer(sample["input_text"], return_tensors="pt", max_length=1024, truncation=True).to(device)

    with torch.no_grad():
        out_ids = best_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            length_penalty=1.5,
            repetition_penalty=1.0,
            early_stopping=True
            )

    pred = bart_tokenizer.decode(out_ids[0], skip_special_tokens=True)
    print(f"\n Game: {sample['game']}")
    print(f"Ground Truth: {sample['target_text']}")
    print(f"Prediction: {pred}")


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: FaceRig
Ground Truth: Aspects: [Gameplay:characters:negative]
Prediction: Aspects: [Gameplay:characters:positive]

 Game: BioShock 2 Remastered
Ground Truth: Aspects: [Performance:fps:positive]
Prediction: Aspects: [Performance:fps:neutral]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Total Annihilation
Ground Truth: Aspects: [Gameplay:multiplayer:neutral]
Prediction: Aspects: [Gameplay:multiplayer:negative]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Total Annihilation
Ground Truth: Aspects: [Gameplay:graphics:positive, Gameplay:price:positive]
Prediction: Aspects: [Gameplay:graphics:negative, Gameplay:price:positive]


... (2 outputs left out)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: Total Annihilation
Ground Truth: Aspects: [Gameplay:multiplayer:neutral]
Prediction: Aspects: [Gameplay:multiplayer:negative]


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Game: VoidExpanse
Ground Truth: Aspects: [Gameplay:gameplay:positive, Gameplay:graphics:positive]
Prediction: Aspects: [Gameplay:gameplay:positive, Gameplay:graphics:positive]

 Game: VoidExpanse
Ground Truth: Aspects: [Gameplay:campaign:neutral]
Prediction: Aspects: [Gameplay:campaign:neutral]


## Export the JSON File

In [21]:
def parse_predicted_aspects(pred_text):
    cleaned = re.sub(r"Aspects\s*:\s*\[?", "", pred_text, flags=re.IGNORECASE)
    cleaned = cleaned.replace("]", "")

    matches = re.findall(r"([A-Za-z0-9_/]+):([^:\],]+):([A-Za-z]+)", cleaned)
    aspects_list = []
    for cat, asp, sent in matches:
        aspects_list.append({
            "category": cat.strip(),
            "aspect": asp.strip(),
            "sentiment": sent.strip().capitalize()
        })
    return aspects_list

In [24]:
with open(JSON_OUTPUT_FILE, "r", encoding="utf-8") as f:
    games_data = json.load(f)


flat_items = []
for game_id, data in games_data.items():
    for review in data.get("reviews", []):
        flat_items.append((game_id, review))

pred_output = {}
best_model.eval()
all_preds = []
batch_size = 128

for i in range(0, len(flat_items), batch_size):
    batch = flat_items[i : i + batch_size]
    prompts = [build_prompt(rev) for _, rev in batch]

    inputs = bart_tokenizer(
        prompts, return_tensors="pt", padding=True, max_length=1024, truncation=True
    ).to(device)

    with torch.no_grad():
        out_ids = best_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            repetition_penalty=1.0,
            length_penalty=1.5,
            early_stopping=True
        )

    pred_text = bart_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
    all_preds.extend(pred_text)

pred_output = {
    game_id: {
        "app_name": data.get("app_name", ""),
        "reviews": data.get("reviews", []),
        "aspects": [],
        "raw_prediction": []
    }
    for game_id, data in games_data.items()
}

seen_per_game = {game_id: set() for game_id in games_data}

for (game_id, review), pred_text in zip(flat_items, all_preds):
    pred_output[game_id]["raw_prediction"].append(pred_text)
    for item in parse_predicted_aspects(pred_text):
        if item["aspect"].lower() not in review.lower():
            continue
        key = (item["category"].lower(), item["aspect"].lower())
        if key not in seen_per_game[game_id]:
            seen_per_game[game_id].add(key)
            pred_output[game_id]["aspects"].append(item)



[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [25]:
pred_output_file = os.path.join(BASE_DIR, "data", "modeled" , "keywords_predicted.jsonl")
os.makedirs(os.path.dirname(pred_output_file), exist_ok=True)
with open(pred_output_file, "w", encoding="utf-8") as f:
    json.dump(pred_output, f, indent=4, ensure_ascii=False)

### Save and Load the Model

In [26]:
best_model_dir = os.path.join(BASE_DIR, "models", "bart_model")
os.makedirs(best_model_dir, exist_ok=True)

best_model.save_pretrained(best_model_dir)
bart_tokenizer.save_pretrained(best_model_dir)

print(f"Best model exported to: {best_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model exported to: /content/IF5153_NLP_Final_Projects/models/bart_model


In [27]:
# Load the Model
bart_tokenizer = BartTokenizer.from_pretrained(best_model_dir)
best_model = BartForConditionalGeneration.from_pretrained(best_model_dir).to(device)

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

In [29]:
meta_path = os.path.join(best_model_dir, "best_score.json")
current_f1 = bart_hyperparametered_task_adapted_evaluation["F1-Score (%)"]

prev_f1 = 0.0
if os.path.exists(meta_path):
    with open(meta_path, "r") as f:
        prev_f1 = json.load(f).get("f1", 0.0)

if current_f1 > prev_f1:
    os.makedirs(best_model_dir, exist_ok=True)
    best_model.save_pretrained(best_model_dir)
    bart_tokenizer.save_pretrained(best_model_dir)
    with open(meta_path, "w") as f:
        json.dump({"f1": current_f1}, f)
    print(f"New best model saved! (F1: {prev_f1}% -> {current_f1}%)")
else:
    print(f"Skipped saving: current F1 ({current_f1}%) did not beat saved best ({prev_f1}%).")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best model saved! (F1: 70.2% -> 74.85%)
